# Prediction using mrcnn-fpn model

reference:
- Model: Mask R-CNN, Kaiming He and Georgia Gkioxari and Piotr Dollár and Ross Girshick,2018, arXiv,cs.CV,https://arxiv.org/abs/1703.06870
- Model: Benchmarking detection transfer learning with vision transformers. Y Li, S Xie, X Chen, P Dollar, K He, R Girshick
arXiv preprint arXiv:2111.11429, 2021


In [10]:
import torch
import os
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2,MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import io
from tqdm import tqdm
import matplotlib.pyplot as plt
from src.load_mrcnn import *

In [11]:
GPU_ID = 0 # GPUnumber to be used
DEVICE = torch.device(f'cuda:{GPU_ID}' if torch.cuda.is_available() else 'cpu')

image_size = 256  # choice is 256 or 512
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 5e-4
BATCH_SIZE = 8 # choice is 64 or 8
NUM_EPOCHS = 100
NUM_WORKERS = 4
CHECKPOINT_FILE = "256_multispecies_777_epoch.tar"
PIN_MEMORY = True
SAVE_MODEL = False
LOAD_MODEL = True

TEST_DIR = 'dataset/png_'+str(image_size)+'/image/test'
IMAGE_SIZE = [256,256] # [512, 512] for 512 data

#### model used is the improved version of Mask-RCNN-FPN.v2

In [12]:
def get_model():
    model = maskrcnn_resnet50_fpn_v2(weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT,\
                                     box_detections_per_img = 200) #pretrained=True
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(
            in_features, num_classes=1+4) 
    model.to(DEVICE)
    
    return model

In [13]:
def save_checkpoint(state, filename='./checkpoint/'+str(image_size)+'_mask_rcnn.pth.tar'):
    print("=> Model improved, Saving checkpoint")
    torch.save(state, filename)

In [14]:
def load_checkpoint(checkpoint, model, optimizer, lr):
    print("=> Loading checkpoint")
    model.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])

    # If we don't do this then it will just have learning rate of old checkpoint
    # and it will lead to many hours of debugging \:
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

In [15]:
model = get_model()
optimizer = torch.optim.AdamW(params=model.parameters(),
                                  lr=LEARNING_RATE,
                                  weight_decay=WEIGHT_DECAY
                                  )

In [16]:
if LOAD_MODEL and CHECKPOINT_FILE in os.listdir('./model'):
    print("Loading checkpoint")
    c_f = os.path.join('./model',CHECKPOINT_FILE)
    load_checkpoint(torch.load(c_f), model, optimizer, LEARNING_RATE)

Loading checkpoint


/tmp/ipykernel_2220316/1593808751.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  load_checkpoint(torch.load(c_f), model, optimizer, LEARNING_RATE)


=> Loading checkpoint


def predict_single_frame(frame):
    images = cv2.resize(frame, IMAGE_SIZE, cv2.INTER_LINEAR)/255
    images = torch.as_tensor(images, dtype=torch.float32).unsqueeze(0)
    images = images.swapaxes(1, 3).swapaxes(2, 3)
    images = list(image.to(DEVICE) for image in images)
    
    with torch.no_grad():
      pred = model(images)

    # color information 
    #colors = [(0,0,0),(1,0,0),(0,1,0),(1,1,1),(0,0,1)]
    cell_color = [(0,0,0),(1,159/255,155/255),(144/255,229/255,161/255),\
                  (161/255,201/255,244/255),(1,180/255,130/255)]
    
    im = images[0].swapaxes(0, 2).swapaxes(0, 1).detach().cpu().numpy().astype(np.float32)
    im2 = np.zeros_like(im).astype(np.float32)
    
    msk=pred[0]['masks'].detach().cpu().numpy()
    scr=pred[0]['scores'].detach().cpu().numpy()
    box=pred[0]['boxes'].detach().cpu().numpy()
    cell_ids=pred[0]['labels'].detach().cpu().numpy()
    
    dfx = pd.DataFrame()
    dfx['labels'] = cell_ids
    dfx['scores'] = scr
    dfx['width'] = [x2-x1 for x1,y1,x2,y2 in list(box)]
    dfx['height'] = [y2-y1 for x1,y1,x2,y2 in list(box)]
    area_w = [np.unique(k.flatten()>0.3, return_counts=True) for k in msk]
    area = [j[1][1] if len(j[1]) > 1 else 0 for j in area_w]

    #area = [j[1][1] for j in area_w] #本物
    dfx['area'] = area


    #cell_types = ['stuff', 'uco']
    cell_types = ['stuff', 'vessel','ray','fiber', 'parenchyma']
    
    cell_type = [cell_types[n] for n in cell_ids]
    dfx['cell_name']= cell_type
    
    for i in range(len(pred[0]['masks'])):
        msk=pred[0]['masks'][i,0].detach().cpu().numpy()
        scr=pred[0]['scores'][i].detach().cpu().numpy()
        box=pred[0]['boxes'][i].detach().cpu().numpy()
        cell_name=pred[0]['labels'][i].detach().cpu().numpy()
        
        if scr>0.3 :  #0.9
            #cv2.rectangle(im, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), colors[cell_type], 2)
            cv2.putText(im, "{0:.2f}%".format(scr*100), (int(box[0]+5), int(box[1])+15), cv2.FONT_HERSHEY_SIMPLEX,
                        0.3, (0,0,0), 1, cv2.LINE_AA)
            im2[:, :, :][msk > 0.37] = cell_color[cell_name]


    return (cv2.addWeighted(im, 0.6, im2, 0.4, 0)*255).astype(np.uint8),dfx, pred

def circle_plot(df):
    category_totals = df.groupby('cell_name')['area'].sum().reset_index()
    
    labels = category_totals['cell_name']
    sizes = category_totals['area']
    u, counts = np.unique(df['cell_name'], return_counts=True)
    colors = sns.color_palette('pastel')[0:len(labels)]
    
    items = [labels.values, labels]
    amount = [sizes.values, counts]
    comment = ['Occupied area of each cell','Number of each cell']
    
    fig, ax = plt.subplots(2,1,figsize=(4,4))
    for j in range(2):
        ax[j].pie(amount[j], labels=items[j], colors=colors, autopct='%1.1f%%', startangle=140)
        ax[j].axis('equal') 
        ax[j].set_title(comment[j])
    plt.tight_layout()
    plt.show()

def image_circle_plot(result,df, save_data=False):
    category_totals = df.groupby('cell_name')['area'].sum().reset_index()
    
    labels = category_totals['cell_name']
    sizes = category_totals['area']
    u, counts = np.unique(df['cell_name'], return_counts=True)
    colors = sns.color_palette('pastel')[0:len(labels)]
    
    items = [labels.values, labels]
    amount = [sizes.values, counts]
    comment = ['Occupied area of each cell','Number of each cell']

    plt.figure(figsize=(8,4))
    plt.subplot(121)
    plt.imshow(result)
    plt.axis('off')
    plt.subplot(222)
    plt.pie(amount[0], labels=items[0], colors=colors, autopct='%1.1f%%', startangle=140)
    plt.axis('equal') 
    plt.title(comment[0],fontsize=10)
    plt.subplot(224)
    plt.pie(amount[1], labels=items[1], colors=colors, autopct='%1.1f%%', startangle=140)
    plt.axis('equal') 
    plt.title(comment[1],fontsize=10)
    plt.tight_layout()
    if save_data == True:
        buf = io.BytesIO() # prepare buffer
        plt.savefig(buf, format='png') # store in the buffer
        enc = np.frombuffer(buf.getvalue(), dtype=np.uint8) # read from buffer
        dst = cv2.imdecode(enc, 1) # decoding
        dst = dst[:,:,::-1] # BGR->RGB
        plt.close()
        return dst
    else:
        plt.show()
    

In [17]:
import numpy as np
import torch
import cv2
from skimage.segmentation import expand_labels

def predict_multi_frame(model, frame, IMAGE_SIZE, score=0.7, msk_prob=0.7,
                        DEVICE="cuda" if torch.cuda.is_available() else "cpu"):
    # === 入力画像の前処理 ===
    images = cv2.resize(frame, IMAGE_SIZE, cv2.INTER_LINEAR) / 255.0
    images = torch.as_tensor(images, dtype=torch.float32).unsqueeze(0)  # [1, H, W, C]
    images = images.permute(0, 3, 1, 2)  # [1, C, H, W]
    images = [image.to(DEVICE) for image in images]

    # === 推論 ===
    with torch.no_grad():
        pred = model(images)

    # === ラベル→グレイ値マッピング定義 ===
    label_to_gray = {1: 63, 2: 127, 3: 191, 4: 255}

    # === 空のラベル画像を作成（整数IDとして保持）===
    height, width = images[0].shape[1:]
    im_label = np.zeros((height, width), dtype=np.uint8)

    for i in range(len(pred[0]['masks'])):
        mask_tensor = pred[0]['masks'][i]
        score_val = pred[0]['scores'][i].item()
        label_val = pred[0]['labels'][i].item()

        # スコアしきい値と対象ラベルのみ使用
        if score_val > score and label_val in label_to_gray:
            msk = mask_tensor[0].detach().cpu().numpy() if mask_tensor.ndim == 3 else mask_tensor.detach().cpu().numpy()
            im_label[msk > msk_prob] = label_val

    # === ラベルを拡張（整数で安全に拡張）===
    im_label_expanded = expand_labels(im_label, distance=1)

    # === グレイレベルに変換（明示的かつ安全）===
    vectorized = np.vectorize(lambda x: label_to_gray.get(x, 0))
    im_gray = vectorized(im_label_expanded).astype(np.uint8)

    return im_gray


In [18]:
from PIL import Image
import pandas as pd
import seaborn as sns


model.train(False)

i = 2

testfilename = [fl for fl in os.listdir(TEST_DIR) if fl.endswith('.png')]

print('You have ', str(len(testfilename)), \
      ' test images, and you selected image No. ', str(i))

img0 = Image.open(os.path.join(TEST_DIR,testfilename[i])).convert('RGB')
img0 = np.array(img0)
img = cv2.imread(os.path.join(TEST_DIR,testfilename[i]))

result, dfx, pred = predict_single_frame(img)
image_circle_plot(result, dfx)

You have  108  test images, and you selected image No.  2


NameError: name 'IMAGE_SIZE' is not defined

The end of script.

an appendix for gifmovie making

- making circular scan of the large image

In [ ]:
'''
imx = Image.open('./dataset/target/Q_acuta_prep_11143_004_3600.png').convert('RGB')

# moving range
theta = 360
rad = np.deg2rad(theta)
drad = rad/360
r = 500

images = []
count = 0
for i in tqdm(range(360)):
    count += 1
    rsinθ = r * np.sin(-drad*count)
    rcosθ = r * np.cos(-drad*count)
    x = rcosθ
    y = rsinθ
    imxp = imx.crop((1250+x, 1250+y, 1250+x+image_size, 1250+y+image_size))
    result_frame, df, _ = predict_single_frame(np.array(imxp))
    dst = image_circle_plot(result_frame, df, save_data=True)
    images.append(dst)

import imageio
imageio.mimsave('./output/result'+str(image_size)+'circle.gif', images,'GIF', fps=5)
'''

 show segmented and detected mask area 

In [ ]:
image_size

In [ ]:
ls dataset/target/*_msk.png


In [ ]:
import glob

# 最新状態で再取得（マスクは含まれないはず）
flns = sorted(glob.glob("dataset/target/*.png"))

for f in flns:
    print(f)


In [ ]:
print(os.getcwd())
import glob

# パスのリストを取得
flns = sorted(glob.glob("dataset/target/img/*.png"))

# 結果確認
print(f"Found {len(flns)} PNG files.")
for f in png_files:
    print(f)

In [ ]:

fl_sel = 1


selected_path = flns[fl_sel]
fln = os.path.splitext(os.path.basename(selected_path))[0]
imx = Image.open(selected_path).convert('RGB')


# moving range


images = []
count = 0
for i in tqdm(range(418)):
    count += 1
    imxp = imx.crop((1250, 3348-i*8, 1250+image_size, 3348-i*8+image_size))
    result_frame, df, _ = predict_single_frame(np.array(imxp))
    dst = image_sequence(result_frame, save_data=True)
    images.append(dst)

import imageio
imageio.mimsave('./output/radial_scan_'+fln+'_'+str(image_size)+'.gif', images,'GIF', fps=5)


In [ ]:
fln

In [ ]:
'''
result, _, pred = predict_single_frame(img)
plt.imshow(result)
plt.axis('off')
plt.show()


each = pred[0]['masks'][0].detach().cpu().numpy()[0]
each_mask = np.zeros_like(img0)
for i in range(3):
    each_mask[:,:,i] = each > 0.3
img_out = img0*each_mask
plt.imshow(img_out) 
plt.axis('off')
plt.show()
'''

The end of script.